In [23]:
with open('KJV-Bible.txt', 'r', encoding='utf-8') as f:
    text = f.read() 

In [24]:
#get unique characters in text
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !"#&'(),-.0123456789:;<>?ABCDEFGHIJKLMNOPQRSTUVWXYZ[]abcdefghijklmnopqrstuvwxyz
81


In [25]:
#tokenize characters (char->int)
str_to_int = { ch:i for i,ch in enumerate(chars) }
int_to_str = { i:ch for i,ch in enumerate(chars) }
def encode(s):
    return [str_to_int[c] for c in s]
def decode(l):
    return ''.join([int_to_str[i] for i in l])

print(encode("test"))
print(decode(encode("test")))

[74, 59, 73, 74]
test


In [26]:
# encode the entire text and store in Tensor
import torch
data = torch.tensor(encode(text), dtype = torch.long)
print(data.shape, data.dtype)
print(data[:1000])

torch.Size([4363228]) torch.int64
tensor([46, 34, 31,  1, 34, 41, 38, 51,  1, 28, 35, 28, 38, 31,  0,  0, 29, 41,
        40, 46, 27, 35, 40, 35, 40, 33,  1, 46, 34, 31,  1, 41, 38, 30,  1, 27,
        40, 30,  1, 40, 31, 49,  1, 46, 31, 45, 46, 27, 39, 31, 40, 46, 45,  0,
         0, 46, 44, 27, 40, 45, 38, 27, 46, 31, 30,  1, 41, 47, 46,  1, 41, 32,
         1, 46, 34, 31,  1, 41, 44, 35, 33, 35, 40, 27, 38,  1, 46, 41, 40, 33,
        47, 31, 45,  1, 27, 40, 30,  1, 49, 35, 46, 34,  1, 46, 34, 31,  1, 32,
        41, 44, 39, 31, 44,  1, 46, 44, 27, 40, 45, 38, 27, 46, 35, 41, 40, 45,
         1, 30, 35, 38, 35, 33, 31, 40, 46, 38, 51,  1, 29, 41, 39, 42, 27, 44,
        31, 30,  1, 27, 40, 30,  1, 44, 31, 48, 35, 45, 31, 30,  1, 28, 51,  1,
        34, 35, 45,  1, 39, 27, 36, 31, 45, 46, 51,  6, 45,  1, 45, 42, 31, 29,
        35, 27, 38,  1, 29, 41, 39, 39, 27, 40, 30,  0,  0, 27, 42, 42, 41, 35,
        40, 46, 31, 30,  1, 46, 41,  1, 28, 31,  1, 44, 31, 27, 30,  1, 35, 40,
      

In [27]:
# split data into training and validation sets
n = int(0.9 * len(data))
train = data[:n]
test = data[n:]

block_size = 8
train[:block_size+1]

tensor([46, 34, 31,  1, 34, 41, 38, 51,  1])

In [28]:
x = train[:block_size]
y = train[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target: {target}")

when input is tensor([46]) the target: 34
when input is tensor([46, 34]) the target: 31
when input is tensor([46, 34, 31]) the target: 1
when input is tensor([46, 34, 31,  1]) the target: 34
when input is tensor([46, 34, 31,  1, 34]) the target: 41
when input is tensor([46, 34, 31,  1, 34, 41]) the target: 38
when input is tensor([46, 34, 31,  1, 34, 41, 38]) the target: 51
when input is tensor([46, 34, 31,  1, 34, 41, 38, 51]) the target: 1


In [29]:
batch_size = 4 # number of independent sequences to process in parallel
block_size = 8 # maximum context length

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train if split == 'train' else test
    ix = torch.randint(len(data) - block_size, (batch_size,))
    # get x block
    x = torch.stack([data[i:i+block_size] for i in ix])
    # get y block (1 offset from x)
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('-----')

for b in range(batch_size): # batch dimension
    for t in range(block_size): # time dimension
        # x and y batches are already offset so we dont offset them here
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target: {target}")

inputs:
torch.Size([4, 8])
tensor([[72, 59, 54,  1, 68, 55, 72, 72],
        [59,  1, 73, 62, 55, 66, 66,  1],
        [62, 69, 66, 58,  9,  1, 69, 60],
        [63, 68, 11,  1, 46, 62, 63, 73]])
targets:
torch.Size([4, 8])
tensor([[59, 54,  1, 68, 55, 72, 72, 69],
        [ 1, 73, 62, 55, 66, 66,  1, 58],
        [69, 66, 58,  9,  1, 69, 60,  1],
        [68, 11,  1, 46, 62, 63, 73,  1]])
-----
when input is [72] the target: 59
when input is [72, 59] the target: 54
when input is [72, 59, 54] the target: 1
when input is [72, 59, 54, 1] the target: 68
when input is [72, 59, 54, 1, 68] the target: 55
when input is [72, 59, 54, 1, 68, 55] the target: 72
when input is [72, 59, 54, 1, 68, 55, 72] the target: 72
when input is [72, 59, 54, 1, 68, 55, 72, 72] the target: 69
when input is [59] the target: 1
when input is [59, 1] the target: 73
when input is [59, 1, 73] the target: 62
when input is [59, 1, 73, 62] the target: 55
when input is [59, 1, 73, 62, 55] the target: 66
when input is [59,

In [30]:
print(xb)

tensor([[72, 59, 54,  1, 68, 55, 72, 72],
        [59,  1, 73, 62, 55, 66, 66,  1],
        [62, 69, 66, 58,  9,  1, 69, 60],
        [63, 68, 11,  1, 46, 62, 63, 73]])


In [34]:
import torch
import torch.nn as nn
from torch.nn import functional as F

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):

        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx) # (B,T,C). Each index replaced by row of scores

        loss = None
        if targets is not None:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss
    
    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            logits, loss = self(idx)
            logits = logits[:, -1, :] # (B,T,C) -> (B,C), last element in time dimension
            probs = F.softmax(logits, dim=-1) # (B, C)
            idx_next = torch.multinomial(probs, num_samples=1) # sample from probabilities (B, 1)
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx


m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

print(decode(m.generate(idx = torch.zeros((1,1), dtype=torch.long), max_new_tokens=100)[0].tolist()))


torch.Size([32, 81])
tensor(5.0759, grad_fn=<NllLossBackward0>)

v-1h-Oy9Ui[RG<r-VpbN[?FZE? :qVM3s6fW6ZszD!x:5#OPC51qDmU#Ee6u
.]!Z3:o<"a-i?aIT:0!#
-u<jZX?O8ij,:r4R)O


In [36]:
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [39]:
batch_size = 32
for steps in range(10000):

    #sample a batch of data
    xb, yb = get_batch('train')

    #evaluate the loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

print(loss.item())

2.297921895980835


In [41]:
print(decode(m.generate(idx = torch.zeros((1,1), dtype=torch.long), max_new_tokens=300)[0].tolist()))



3 s Gid thofothe] se ori, Andond Andlave me he>N'sof Je ay Te rypl the? ud theatherixcasund un, shthe be, ane ungre h Init wag s, the, d Anthaf de f [If ume ai?Tof opteald, nse, rsw's in ws.
1 f s, foun s haf t bdathais.)ve gedem rathainem] m: and My Abuthet thee id [th y fthat theand sam d So thes.
